# Fase 1 — Base longitudinal por centro (2025 ↔ 2026)

Reporte **único** de la fase 1 del pipeline (`estructura.py`). Documenta, para **docentes** y **estudiantes**, qué se filtró para dejar datasets aptos para análisis a través del tiempo.

---

## Qué hace el pipeline con los datasets crudos

Entrada: los cuatro Excel de la carpeta de datos (`datosUCU{2025,2026}_{estu,doc}.xlsx`).

### Paso A — Limpieza (`limpiar` → `processed/<tipo>_<anio>_clean.csv`)

Sobre cada base cruda, de forma independiente por tipo/año:

| Cambio | Detalle |
|---|---|
| Tipos de días | `Dias4/5/6` → numérico; flag si salen del rango del mes (no recorta) |
| Texto | trim en columnas string |
| Vulnerabilidad | `IVSMEDIA` / `CONTEXTO` → `ivsmedia_q`, `contexto_q`, `contexto_zona`, y eje único `vuln_q` |
| Uso | `dias_totales` (suma abr+may+jun) y `accedio` (`dias_totales > 0`) |
| Centro | `ID_CENTRO` → `ID_CENTRO_estudiantes` o `ID_CENTRO_docentes` |
| Metadatos | columnas `tipo`, `anio` |

**Devuelve (por tipo/año):** un DataFrame limpio al grano original (asignación en docentes, alumno en estudiantes), guardado como CSV en `processed/`.

### Paso B — Persistencia longitudinal (`aplicar_persistencia_ambos_tipos`)

Sobre los cuatro limpios en memoria:

| Criterio | Detalle |
|---|---|
| Por tipo | se conservan filas cuyo centro aparece en **2025 y 2026** |
| Cruce (default) | si `CRUZAR_TIPOS_PERSISTENCIA = True`, el centro también debe ser persistente en el **otro** tipo (intersección simétrica) |

**Devuelve / escribe en la carpeta `reportes/` junto a los datos:**

| Archivo | Contenido |
|---|---|
| `<tipo>_2025_persistente.csv` / `<tipo>_2026_persistente.csv` | filas cuyo centro pasó el filtro |
| `<tipo>_filas_eliminadas.csv` | filas descartadas (auditoría) |
| `impacto_persistencia_<tipo>.csv` | conteos y % vs limpio |
| `centros_eliminados_<tipo>.csv` | detalle por centro eliminado |
| `impacto_persistencia_fase1.csv` | impacto de ambos tipos juntos |
| `resumen_fase1.txt` | resumen legible consolidado |

`colapsar_docentes` y `panel` existen en el módulo pero **no** se corren en `main()` de esta fase.

---

## Criterio de persistencia (idéntico en ambos tipos)

Un centro solo sirve si está presente en **ambos** años. Además (flag `CRUZAR_TIPOS_PERSISTENCIA = True` en `estructura.py`), el código de centro debe ser persistente también del **otro** tipo. Eso produce la **intersección simétrica** de centros: el mismo conjunto se conserva en docentes y en estudiantes.

- Se **conservan** las filas cuyo `ID_CENTRO_<tipo>` está en ese conjunto → base longitudinal (`*_persistente.csv`).
- Se **eliminan** las filas de centros que aparecen en un solo año (o no cruzan al otro tipo) → quedan en `*_filas_eliminadas.csv` para auditoría.

Este notebook **no** reimplementa el filtro: llama a `estructura.aplicar_persistencia_ambos_tipos` y muestra el impacto de forma legible.


In [ ]:
import pathsetup  # raíz del repo en sys.path (notebooks en reportes/)
from pathlib import Path

import pandas as pd

from compare_datasets_generic import cargar
from estructura import (
    CARPETA_DATOS,
    SALIDA,
    REPORTES,
    CRUZAR_TIPOS_PERSISTENCIA,
    aplicar_persistencia_ambos_tipos,
)

print("Carpeta datos :", CARPETA_DATOS)
print("Limpíos       :", SALIDA)
print("Reportes      :", REPORTES)
print("Cruce tipos   :", CRUZAR_TIPOS_PERSISTENCIA)


## 1. Cargar limpios y (re)aplicar persistencia

Si los CSV limpios ya existen, se cargan y se regeneran los persistentes + tablas de impacto en `reportes/`. Para regenerar también los limpios desde el `.xlsx`, correr `python estructura.py`.

In [ ]:
limpios = {}
for tipo in ("docentes", "estudiantes"):
    for anio in (2025, 2026):
        path = SALIDA / f"{tipo}_{anio}_clean.csv"
        limpios[(tipo, anio)] = cargar(path)
        print(f"{tipo} {anio}: {len(limpios[(tipo, anio)]):,} filas")

resultados = aplicar_persistencia_ambos_tipos(limpios)

## 2. Impacto: filas eliminadas vs DF limpio

Porcentaje de filas eliminadas / persistentes respecto al limpio (por año y total). Misma tabla consolidada que `reportes/impacto_persistencia_fase1.csv`.

In [ ]:
impacto = pd.concat(
    [resultados["docentes"]["impacto"], resultados["estudiantes"]["impacto"]],
    ignore_index=True,
)
impacto

In [ ]:
# Lectura amigable: solo totales por tipo
totales = impacto[impacto["anio"] == "total"].copy()
for _, r in totales.iterrows():
    print(
        f"{r['tipo']:>12}: eliminadas {int(r['filas_eliminadas']):,} / {int(r['filas']):,} "
        f"({r['%_eliminadas']}%)  |  persistentes {int(r['filas_persistentes']):,} "
        f"({r['%_persistentes']}%)"
    )

## 3. Centros conservados vs no conservados

Mismo conjunto de centros persistentes en ambos tipos cuando el cruce está activo (intersección).

In [ ]:
for tipo in ("docentes", "estudiantes"):
    res = resultados[tipo]
    print(
        f"{tipo}: centros conservados={len(res['centros_ambos'])} | "
        f"no conservados={res['n_centros_no_conservados']}"
    )

c_doc = resultados["docentes"]["centros_ambos"]
c_est = resultados["estudiantes"]["centros_ambos"]
print(f"\nMisma interseccion de centros (docentes == estudiantes): {c_doc == c_est}")
print(f"Cardinalidad: {len(c_doc)}")

## 4. Qué se borró (auditoría)

Muestra de filas eliminadas y detalle por centro (año + código + n filas).

In [ ]:
for tipo in ("docentes", "estudiantes"):
    elim = resultados[tipo]["eliminadas"]
    col = resultados[tipo]["col_centro"]
    print(f"=== {tipo} — {len(elim):,} filas eliminadas ===")
    print(elim["anio"].value_counts().sort_index().to_string())
    print(f"Centros distintos eliminados: {elim[col].nunique()}")
    display(elim.head(5))
    display(resultados[tipo]["detalle_centros"].head(10))
    print()

## 5. Resumen final

Texto idéntico a `reportes/resumen_fase1.txt`.


In [ ]:
resumen_path = REPORTES / "resumen_fase1.txt"
print(resumen_path.read_text(encoding="utf-8"))
print("Archivos en reportes/:")
for f in sorted(REPORTES.iterdir()):
    if f.is_file() and not f.name.startswith("."):
        print(f"  - {f.name}")


## Cómo regenerar todo

```bash
# Desde la raíz del repo (usa el venv del proyecto):
.venv/bin/python estructura.py
```

Eso reescribe los `*_clean.csv` en `processed/` y los `*_persistente.csv` + auditoría en `reportes/`. Este notebook solo reporta; la fuente de verdad de las transformaciones es `estructura.py`.
